# LAB 03 — Word Embeddings: From Co-occurrence Matrix to Word2Vec


In [1]:
import sys
import os
import time
import math
import numpy as np
import pandas as pd
from collections import Counter

sys.path.append(os.path.abspath("."))
from cooccurrence import (
    build_vocabulary,
    build_cooccurrence_matrix,
    cosine_similarity,
    most_similar,
    word_similarity,
    compute_matrix_stats
)


## 10. Experiment 1 — Word-context representation & 11. Core implementation

Khảo sát ma trận đồng xuất hiện $X \in \mathbb{R}^{|V| \times |V|}$ trên tập ngữ liệu đại diện với các kích thước cửa sổ context khác nhau ($w = 1, 2, 5$).  
So sánh:
- Vocabulary size ($|V|$)
- Matrix shape và kích thước
- Number of non-zero values và Sparsity (%)
- Độ tương đồng Cosine giữa các cặp từ: `doctor - hospital`, `doctor - physician`, `cat - dog`.


In [2]:
sentences = [
    "the doctor examines the sick patient in the hospital",
    "the physician prescribes medicine for the patient",
    "a doctor and nurse provide medical treatment in the clinical center",
    "the patient received clinical therapy for the chronic disease",
    "the hospital treats patient suffering from serious disease",
    "clinical therapy helps patient recover from severe disease",
    "the physician and doctor discuss medical treatment options",
    "the nurse cares for the patient in the hospital ward",
    "effective treatment cured the rare disease in the clinic",
    "doctors and physicians work together in modern hospitals",
    "the doctor diagnosed the disease and admitted patient to hospital",
    "physicians recommend hospital care and clinical therapy for patient",
    "the king rules the royal kingdom with the queen",
    "the wise king is a powerful man",
    "the noble queen is a gracious woman",
    "the prince is the son of the king and queen",
    "the monarch wore a golden crown in the palace",
    "the king and queen sat upon the royal throne",
    "the queen welcomed the king back to the palace",
    "the man became king while the woman became queen",
    "the cat plays with the small dog in the garden",
    "a domestic cat likes to eat fresh fish",
    "the loyal dog barks and guards the house",
    "the dog and cat are friendly domestic pets",
    "the cat chases mice while the dog sleeps",
    "a puppy is a young dog and a kitten is a young cat",
    "the car drives fast on the highway road",
    "the automobile needs fuel and engine repair",
    "the driver parked the car near the automobile shop",
    "a modern automobile is an efficient road vehicle",
    "the car and automobile travel across the city",
    "the fast car overtook the heavy automobile",
    "the computer executes software program and code",
    "engineers develop software algorithms on the computer",
    "the computer processor calculates complex machine data",
    "the machine learns patterns from large computer data",
    "programmers write computer code for software systems",
    "the football player kicked the ball into the goal",
    "fans watch the football match in the large stadium",
    "the football team won the championship game",
    "players run across the football field during the game",
    "the striker scored a goal in the football match",
    "monkeys eat ripe yellow banana in the tropical jungle",
    "the sweet banana is a healthy fruit to eat",
    "children enjoy fresh apple and sweet banana",
    "banana and apple are nutritious fresh fruits",
    "the customer deposited money into the bank account",
    "the bank provides financial loans and saves paper money",
    "investors withdraw cash and money from the commercial bank",
    "the central bank manages national money and finance",
    "we sat on the river bank watching the water stream",
    "green grass grows along the muddy river bank",
    "the boat docked at the quiet river bank near the water",
    "clear water flows along the peaceful river bank"
]

corpus = [s.lower().split() for s in sentences]
vocabulary = build_vocabulary(corpus)
print(f"Tổng số câu trong corpus: {len(corpus)}")
print(f"Vocabulary size |V|: {len(vocabulary)}")


Tổng số câu trong corpus: 54
Vocabulary size |V|: 211


In [3]:
window_sizes = [1, 2, 5]
cooc_matrices = {}
cooc_stats = []

for w in window_sizes:
    mat = build_cooccurrence_matrix(corpus, vocabulary, window_size=w)
    cooc_matrices[w] = mat
    stats = compute_matrix_stats(mat)
    sim_dh = word_similarity("doctor", "hospital", mat, vocabulary)
    sim_dp = word_similarity("doctor", "physician", mat, vocabulary)
    sim_cd = word_similarity("cat", "dog", mat, vocabulary)
    cooc_stats.append({
        "Window": w,
        "Matrix Shape": f"{stats['shape'][0]}x{stats['shape'][1]}",
        "Total Elements": stats["total_elements"],
        "Non-zero Values": stats["non_zero"],
        "Sparsity (%)": round(stats["sparsity_percent"], 2),
        "Sim(doctor, hospital)": round(sim_dh, 4),
        "Sim(doctor, physician)": round(sim_dp, 4),
        "Sim(cat, dog)": round(sim_cd, 4)
    })

df_cooc = pd.DataFrame(cooc_stats)
print(df_cooc.to_string(index=False))


 Window Matrix Shape  Total Elements  Non-zero Values  Sparsity (%)  Sim(doctor, hospital)  Sim(doctor, physician)  Sim(cat, dog)
      1      211x211           44521              680         98.47                 0.4629                  0.7071         0.5641
      2      211x211           44521             1206         97.29                 0.5500                  0.6804         0.5934
      5      211x211           44521             2123         95.23                 0.6589                  0.7628         0.7588


In [4]:
for w in window_sizes:
    print(f"\n--- Top-5 Most Similar to 'doctor' (Co-occurrence Window = {w}) ---")
    top_w = most_similar("doctor", cooc_matrices[w], vocabulary, top_k=5)
    for rank, (word, score) in enumerate(top_w, 1):
        print(f"{rank}. {word:12s}: {score:.4f}")



--- Top-5 Most Similar to 'doctor' (Co-occurrence Window = 1) ---
1. guards      : 0.8165
2. physician   : 0.7071
3. king        : 0.6794
4. queen       : 0.6405
5. car         : 0.6172

--- Top-5 Most Similar to 'doctor' (Co-occurrence Window = 2) ---
1. dog         : 0.7945
2. king        : 0.7702
3. car         : 0.7378
4. cares       : 0.7071
5. queen       : 0.7035

--- Top-5 Most Similar to 'doctor' (Co-occurrence Window = 5) ---
1. nurse       : 0.7762
2. physician   : 0.7628
3. in          : 0.7539
4. the         : 0.7312
5. medical     : 0.7281


## 12. Từ Co-occurrence Matrix đến Dense Embedding

Khi kích thước từ vựng tăng lên $|V| = 100,000$ từ:
- Kích thước ma trận: $100,000 \times 100,000 = 10^{10}$ phần tử.
- Bộ nhớ yêu cầu: Với kiểu số thực 32-bit (`float32`), cần $40 \text{ GB}$ RAM chỉ để lưu trữ ma trận thưa.
- Độ thưa (Sparsity) thực tế thường vượt quá $99.9\%$, gây lãng phí bộ nhớ và chi phí tính toán cực kỳ lớn.
- Khả năng khái quát hóa (Generalization) kém: Hai từ đồng nghĩa nếu xuất hiện ở các ngữ cảnh khác nhau sẽ có vector trực giao ($\cos = 0$).

$\implies$ **Động lực:** Nén không gian biểu diễn từ $|V|$ chiều thưa thớt sang không gian vector đặc (dense representation) có số chiều thấp ($d \in [50, 300]$) bằng các mô hình học biểu diễn phân tán (Word2Vec).


## 13, 14, 15, 16. Neural Word Embeddings: Continuous Bag-of-Words (CBOW) & Skip-gram

- **CBOW:** Dự đoán từ trung tâm dựa vào các từ ngữ cảnh ($\text{Context} \to \text{Target}$).
- **Skip-gram:** Dự đoán các từ ngữ cảnh xung quanh dựa vào từ trung tâm ($\text{Target} \to \text{Context}$).
- Triển khai thuật toán **Skip-gram với Negative Sampling (SGNS)** bằng Python thuần và NumPy, không sử dụng thư viện bên ngoài.


In [5]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -6.0, 6.0)))

class KeyedVectors:
    def __init__(self, w2i, i2w, vectors):
        self.key_to_index = w2i
        self.index_to_key = i2w
        self.vectors = vectors
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        self.unit_vectors = vectors / norms

    def __getitem__(self, word):
        if word not in self.key_to_index:
            raise KeyError(f"Word '{word}' not in vocabulary")
        return self.vectors[self.key_to_index[word]]

    def __contains__(self, word):
        return word in self.key_to_index

    def similarity(self, w1, w2):
        if w1 not in self.key_to_index or w2 not in self.key_to_index:
            return 0.0
        return float(np.dot(self.unit_vectors[self.key_to_index[w1]], self.unit_vectors[self.key_to_index[w2]]))

    def most_similar(self, positive=None, negative=None, topn=5):
        if isinstance(positive, str):
            positive = [positive]
        positive = positive or []
        negative = negative or []
        target = np.zeros(self.vectors.shape[1], dtype=np.float32)
        exclude = set(positive + negative)
        for w in positive:
            if w in self.key_to_index:
                target += self.unit_vectors[self.key_to_index[w]]
        for w in negative:
            if w in self.key_to_index:
                target -= self.unit_vectors[self.key_to_index[w]]
        norm = np.linalg.norm(target)
        if norm > 0:
            target /= norm
        sims = np.dot(self.unit_vectors, target)
        ranks = np.argsort(-sims)
        res = []
        for idx in ranks:
            w = self.index_to_key[idx]
            if w not in exclude:
                res.append((w, float(sims[idx])))
            if len(res) >= topn:
                break
        return res

class Word2Vec:
    def __init__(self, sentences, vector_size=100, window=5, min_count=1, epochs=35, negative=5, lr=0.03, seed=42):
        np.random.seed(seed)
        counts = Counter()
        for s in sentences:
            for w in s:
                counts[w] += 1
        vocab = sorted([w for w, c in counts.items() if c >= min_count])
        self.w2i = {w: i for i, w in enumerate(vocab)}
        self.i2w = {i: w for i, w in enumerate(vocab)}
        V = len(vocab)
        D = vector_size
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.epochs = epochs
        self.W_in = (np.random.rand(V, D).astype(np.float32) - 0.5) / D
        self.W_out = np.zeros((V, D), dtype=np.float32)
        
        freqs = np.array([counts[self.i2w[i]]**0.75 for i in range(V)])
        freqs /= freqs.sum()

        for ep in range(epochs):
            curr_lr = lr * max(0.0001, (1.0 - ep / epochs))
            for s in sentences:
                idx_seq = [self.w2i[w] for w in s if w in self.w2i]
                seq_len = len(idx_seq)
                for pos, target in enumerate(idx_seq):
                    start = max(0, pos - window)
                    end = min(seq_len, pos + window + 1)
                    for c_pos in range(start, end):
                        if c_pos == pos:
                            continue
                        ctx = idx_seq[c_pos]
                        negs = np.random.choice(V, size=negative, p=freqs)
                        all_ctx = np.append([ctx], negs)
                        labels = np.zeros(negative + 1, dtype=np.float32)
                        labels[0] = 1.0

                        u_t = self.W_in[target]
                        v_c = self.W_out[all_ctx]
                        scores = np.dot(v_c, u_t)
                        preds = sigmoid(scores)
                        grad = preds - labels

                        grad_u = np.dot(grad, v_c)
                        self.W_out[all_ctx] -= curr_lr * np.outer(grad, u_t)
                        self.W_in[target] -= curr_lr * grad_u

        self.wv = KeyedVectors(self.w2i, self.i2w, self.W_in)


## 17. Experiment 2 — Word2Vec Baseline Training & 18. Inspect Embeddings

Thiết lập siêu tham số chuẩn theo yêu cầu:
- Corpus: 54 câu văn bản thuộc các miền chủ đề đa dạng.
- Vocabulary size: 211 từ.
- Embedding dimension (`vector_size`): 100
- Context window: 5
- Min count: 1
- Epochs: 35
- Training objective: Skip-gram with Negative Sampling (SGNS, $k=5$, negative sampling distribution $U(w)^{3/4}$).


In [6]:
start_t = time.time()
model_base = Word2Vec(
    corpus,
    vector_size=100,
    window=5,
    min_count=1,
    epochs=35,
    negative=5,
    lr=0.03,
    seed=42
)
train_time = time.time() - start_t

print(f"Huấn luyện Word2Vec Baseline hoàn tất trong {train_time:.2f} giây.")
print(f"Số lượng từ vựng trong mô hình: {len(model_base.wv.key_to_index)}")
print(f"Số chiều vector (vector_size): {model_base.vector_size}")
print(f"Kích thước cửa sổ (window): {model_base.window}")


Huấn luyện Word2Vec Baseline hoàn tất trong 4.42 giây.
Số lượng từ vựng trong mô hình: 211
Số chiều vector (vector_size): 100
Kích thước cửa sổ (window): 5


In [7]:
target_words = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]

print("=== 18. INSPECT EMBEDDINGS — TOP-5 MOST SIMILAR WORDS ===")
for word in target_words:
    top5 = model_base.wv.most_similar(word, topn=5)
    print(f"\nTarget Word: [{word.upper()}]")
    for rank, (w, sim) in enumerate(top5, 1):
        print(f"  {rank}. {w:15s} (cosine similarity = {sim:.4f})")


=== 18. INSPECT EMBEDDINGS — TOP-5 MOST SIMILAR WORDS ===

Target Word: [DOCTOR]
  1. nurse           (cosine similarity = 0.9793)
  2. center          (cosine similarity = 0.9748)
  3. recommend       (cosine similarity = 0.9725)
  4. diagnosed       (cosine similarity = 0.9717)
  5. medical         (cosine similarity = 0.9695)

Target Word: [HOSPITAL]
  1. disease         (cosine similarity = 0.9974)
  2. therapy         (cosine similarity = 0.9943)
  3. clinical        (cosine similarity = 0.9858)
  4. for             (cosine similarity = 0.9831)
  5. patient         (cosine similarity = 0.9777)

Target Word: [PATIENT]
  1. for             (cosine similarity = 0.9852)
  2. disease         (cosine similarity = 0.9800)
  3. hospital        (cosine similarity = 0.9777)
  4. therapy         (cosine similarity = 0.9683)
  5. clinical        (cosine similarity = 0.9568)

Target Word: [DISEASE]
  1. hospital        (cosine similarity = 0.9974)
  2. therapy         (cosine similarity = 0.99

## 19. Experiment 3 — Context Window Ablation

Huấn luyện 3 mô hình với các kích thước cửa sổ khác nhau:
- `window = 2`
- `window = 5`
- `window = 10`

So sánh độ tương đồng của các cặp từ đại diện: `doctor - physician`, `doctor - hospital`, `cat - dog`.  
*Câu hỏi nghiên cứu:* Window lớn hơn có luôn tốt hơn không?


In [8]:
model_w2 = Word2Vec(corpus, vector_size=100, window=2, epochs=35, seed=42)
model_w5 = model_base
model_w10 = Word2Vec(corpus, vector_size=100, window=10, epochs=35, seed=42)

window_comparison = [
    {
        "Word Pair": "doctor - physician",
        "Window 2": round(model_w2.wv.similarity("doctor", "physician"), 4),
        "Window 5": round(model_w5.wv.similarity("doctor", "physician"), 4),
        "Window 10": round(model_w10.wv.similarity("doctor", "physician"), 4),
    },
    {
        "Word Pair": "doctor - hospital",
        "Window 2": round(model_w2.wv.similarity("doctor", "hospital"), 4),
        "Window 5": round(model_w5.wv.similarity("doctor", "hospital"), 4),
        "Window 10": round(model_w10.wv.similarity("doctor", "hospital"), 4),
    },
    {
        "Word Pair": "cat - dog",
        "Window 2": round(model_w2.wv.similarity("cat", "dog"), 4),
        "Window 5": round(model_w5.wv.similarity("cat", "dog"), 4),
        "Window 10": round(model_w10.wv.similarity("cat", "dog"), 4),
    }
]

df_win = pd.DataFrame(window_comparison)
print(df_win.to_string(index=False))


         Word Pair  Window 2  Window 5  Window 10
doctor - physician    0.9857    0.9628     0.9514
 doctor - hospital    0.9799    0.8821     0.9364
         cat - dog    0.9688    0.9616     0.9896


## 20. Experiment 4 — Embedding Dimension Ablation

Huấn luyện 3 mô hình với số chiều khác nhau:
- $d = 50$
- $d = 100$
- $d = 300$

Đánh giá các tiêu chí: Training time, Model size (bytes), Similarity scores, Analogy capability.  
*Phân tích đánh đổi:* `dimension` $\leftrightarrow$ `capacity` $\leftrightarrow$ `data` $\leftrightarrow$ `computation`.


In [9]:
dim_configs = [50, 100, 300]
models_dim = {}
dim_eval = []

for d in dim_configs:
    t0 = time.time()
    if d == 100:
        m = model_base
        dur = train_time
    else:
        m = Word2Vec(corpus, vector_size=d, window=5, epochs=35, seed=42)
        dur = time.time() - t0
    models_dim[d] = m
    
    mem_size = m.W_in.nbytes + m.W_out.nbytes
    sim_dp = m.wv.similarity("doctor", "physician")
    sim_kq = m.wv.similarity("king", "queen")
    top_analogy = m.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
    analogy_word = top_analogy[0][0] if top_analogy else "N/A"
    
    dim_eval.append({
        "Dimension (d)": d,
        "Training Time (s)": round(dur, 2),
        "Memory Footprint (KB)": round(mem_size / 1024, 2),
        "Sim(doctor, physician)": round(sim_dp, 4),
        "Sim(king, queen)": round(sim_kq, 4),
        "Top-1 Analogy Result": analogy_word
    })

df_dim = pd.DataFrame(dim_eval)
print(df_dim.to_string(index=False))


 Dimension (d)  Training Time (s)  Memory Footprint (KB)  Sim(doctor, physician)  Sim(king, queen) Top-1 Analogy Result
            50               4.71                  82.42                  0.9484            0.9909                queen
           100               4.42                 164.84                  0.9628            0.9912                queen
           300               4.89                 494.53                  0.9638            0.9885                queen


## 21. Evaluation — Word Similarity Ranking & Comparison with Intuition

Đánh giá định lượng trên danh sách 6 cặp từ bắt buộc:
1. `doctor - physician` (Đồng nghĩa chức nghiệp y tế)
2. `car - automobile` (Đồng nghĩa phương tiện giao thông)
3. `king - queen` (Cặp đối ngẫu hoàng gia)
4. `cat - dog` (Cùng nhóm thú cưng)
5. `doctor - hospital` (Liên quan chủ đề y tế)
6. `computer - banana` (Hai miền ngữ nghĩa hoàn toàn không liên quan)

Xuất kết quả chi tiết ra bảng và ghi lưu vào file `results.csv`.


In [10]:
pairs_to_evaluate = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog"),
    ("king", "queen"),
    ("car", "automobile"),
    ("computer", "banana")
]

results_records = []
for w1, w2 in pairs_to_evaluate:
    s_w2 = model_w2.wv.similarity(w1, w2)
    s_w5 = model_w5.wv.similarity(w1, w2)
    s_w10 = model_w10.wv.similarity(w1, w2)
    s_d50 = models_dim[50].wv.similarity(w1, w2)
    s_d100 = models_dim[100].wv.similarity(w1, w2)
    s_d300 = models_dim[300].wv.similarity(w1, w2)
    
    results_records.append({
        "Word Pair": f"{w1} - {w2}",
        "Window 2": round(s_w2, 4),
        "Window 5": round(s_w5, 4),
        "Window 10": round(s_w10, 4),
        "Dim 50": round(s_d50, 4),
        "Dim 100": round(s_d100, 4),
        "Dim 300": round(s_d300, 4)
    })

df_final_results = pd.DataFrame(results_records)
df_final_results.to_csv("results.csv", index=False)
print(df_final_results.to_string(index=False))


         Word Pair  Window 2  Window 5  Window 10  Dim 50  Dim 100  Dim 300
doctor - physician    0.9857    0.9628     0.9514  0.9484   0.9628   0.9638
 doctor - hospital    0.9799    0.8821     0.9364  0.8866   0.8821   0.8933
         cat - dog    0.9688    0.9616     0.9896  0.9723   0.9616   0.9738
      king - queen    0.9832    0.9912     0.9965  0.9909   0.9912   0.9885
  car - automobile    0.9803    0.8689     0.8752  0.8470   0.8689   0.8765
 computer - banana    0.9901    0.6518     0.4247  0.6026   0.6518   0.7369


## 22 & 23. Evaluation — Word Analogy Vector Arithmetic

Thực hiện phép toán đại số vector:
$$\vec{v} = \vec{v}_{\text{king}} - \vec{v}_{\text{man}} + \vec{v}_{\text{woman}} \approx \vec{v}_{\text{queen}}$$

Khảo sát xem vector mới có tìm được từ `queen` hay không.


In [11]:
analogy_results = model_base.wv.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

print("Phép toán: king - man + woman")
print("Top-5 từ gần nhất với vector kết quả:")
for rank, (word, score) in enumerate(analogy_results, 1):
    print(f"{rank}. {word:15s} (cosine similarity = {score:.4f})")


Phép toán: king - man + woman
Top-5 từ gần nhất với vector kết quả:
1. queen           (cosine similarity = 0.9719)
2. became          (cosine similarity = 0.9284)
3. royal           (cosine similarity = 0.8795)
4. while           (cosine similarity = 0.8783)
5. automobile      (cosine similarity = 0.8757)


## 24. Application — Semantic Search (Liên kết với LAB 01)

- **LAB 01:** Truy vấn bằng TF-IDF và Cosine Similarity (khớp từ vựng chính xác, bị lỗi Lexical Mismatch khi truy vấn dùng từ đồng nghĩa).
- **LAB 03:** Biểu diễn truy vấn và tài liệu bằng Word Embeddings trung bình (Mean Pooling) $\to$ Semantic Search.
- Thử nghiệm truy vấn: `"medical treatment"` trên tập tài liệu không chứa trực tiếp đầy đủ các từ khóa, chứng minh tìm được các tài liệu chứa `therapy`, `clinical`, `patient`.


In [12]:
documents = [
    "the doctor provides clinical therapy and medical treatment for the patient with severe disease",
    "a physician administers drug treatment and patient care inside the hospital",
    "the king rules the royal kingdom with the queen in the palace",
    "the football team won the championship match in the stadium",
    "programmers write computer code for software systems",
    "monkeys eat sweet yellow banana in the tropical jungle",
    "the customer deposited money into the commercial bank account",
    "we sat on the quiet river bank watching the clear water stream"
]

query = ["medical", "treatment"]
q_vec = np.mean([model_base.wv[w] for w in query if w in model_base.wv], axis=0)
q_norm = np.linalg.norm(q_vec)
if q_norm > 0:
    q_vec /= q_norm

search_results = []
for idx, doc in enumerate(documents):
    tokens = [w for w in doc.split() if w in model_base.wv]
    doc_vec = np.mean([model_base.wv[w] for w in tokens], axis=0)
    d_norm = np.linalg.norm(doc_vec)
    if d_norm > 0:
        doc_vec /= d_norm
    sim = float(np.dot(q_vec, doc_vec))
    search_results.append((idx, sim, doc))

search_results.sort(key=lambda x: x[1], reverse=True)

print("=== KẾT QUẢ SEMANTIC SEARCH VỚI QUERY: 'medical treatment' ===")
for rank, (idx, score, doc) in enumerate(search_results, 1):
    print(f"Top {rank} | Doc {idx} | Score: {score:.4f}")
    print(f"       Nội dung: \"{doc}\"\n")


=== KẾT QUẢ SEMANTIC SEARCH VỚI QUERY: 'medical treatment' ===
Top 1 | Doc 1 | Score: 0.9725
       Nội dung: "a physician administers drug treatment and patient care inside the hospital"

Top 2 | Doc 4 | Score: 0.9475
       Nội dung: "programmers write computer code for software systems"

Top 3 | Doc 5 | Score: 0.9357
       Nội dung: "monkeys eat sweet yellow banana in the tropical jungle"

Top 4 | Doc 0 | Score: 0.9326
       Nội dung: "the doctor provides clinical therapy and medical treatment for the patient with severe disease"

Top 5 | Doc 6 | Score: 0.7436
       Nội dung: "the customer deposited money into the commercial bank account"

Top 6 | Doc 7 | Score: 0.6971
       Nội dung: "we sat on the quiet river bank watching the clear water stream"

Top 7 | Doc 3 | Score: 0.6780
       Nội dung: "the football team won the championship match in the stadium"

Top 8 | Doc 2 | Score: 0.6579
       Nội dung: "the king rules the royal kingdom with the queen in the palace"



## 25 & 26. Error Analysis & Polysemy Demonstration

Thực nghiệm đo lường giới hạn của Static Word Embeddings đối với từ đa nghĩa (`bank`):
- `bank` trong ngữ cảnh tài chính (`money`, `financial`)
- `bank` trong ngữ cảnh sông nước (`river`, `water`)
- Xem xét khoảng cách giữa các từ để chứng minh hiện tượng sụp đổ không gian (Polysemy Collapse).


In [13]:
sim_b_money = model_base.wv.similarity("bank", "money")
sim_b_river = model_base.wv.similarity("bank", "river")
sim_b_water = model_base.wv.similarity("bank", "water")
sim_money_river = model_base.wv.similarity("money", "river")

print("=== KHẢO SÁT HIỆN TƯỢNG ĐA NGHĨA (POLYSEMY) CỦA 'BANK' ===")
print(f"Sim(bank, money): {sim_b_money:.4f}")
print(f"Sim(bank, river): {sim_b_river:.4f}")
print(f"Sim(bank, water): {sim_b_water:.4f}")
print(f"Sim(money, river): {sim_money_river:.4f}")


=== KHẢO SÁT HIỆN TƯỢNG ĐA NGHĨA (POLYSEMY) CỦA 'BANK' ===
Sim(bank, money): 0.8502
Sim(bank, river): 0.9792
Sim(bank, water): 0.9790
Sim(money, river): 0.7837
